# RealTianLong · Colab 放大训练

在 Colab GPU 上把 GNN 动态模型（环境 / 角色两个视角）与 RL 角色策略（模仿学习 → PPO）放大训练，
训练入口与本地完全相同（`python -m tianlong.learning.train` / `python -m tianlong.learning.rl.train`），只是参数更大。

1. 菜单 **代码执行程序 → 更改运行时类型 → GPU**（T4 即可）。
2. 仓库若为私有：左侧 🔑 **Secrets** 新建 `GH_TOKEN`（只读 contents 权限的 fine-grained token），并允许本笔记本访问。
3. 依次运行。产物写入 `artifacts/colab/`，最后一格打包下载或存到 Google Drive。

> 训练产物只是**意图的来源**：GNN 预测与 RL 选择都必须经内核结算才成为事实，放大训练不改变这条边界。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo '没有 GPU：GNN 仍可在 CPU 上跑，只是慢'

In [ ]:
import os, subprocess

BRANCH = 'claude/text-game-engine-architecture-lvzxl8'
REPO = 'github.com/jackeywangxinyu/RealTianLong.git'
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f'https://x-access-token:{token}@{REPO}' if token else f'https://{REPO}'
if not os.path.exists('RealTianLong'):
    # 不回显命令行：令牌不进输出
    r = subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', BRANCH, url, 'RealTianLong'],
                       capture_output=True, text=True)
    print('克隆完成' if r.returncode == 0 else '克隆失败：私有仓库请先设置 GH_TOKEN')
%cd RealTianLong
!git log --oneline -1

## 安装
Colab 自带 CUDA 版 torch，满足 `torch>=2.4`，pip 不会重装；只补 PyG、RLlib 与项目本体。

In [ ]:
!pip -q install -e '.[learn,rl,dev]'
import torch, torch_geometric, ray
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| pyg', torch_geometric.__version__, '| ray', ray.__version__)

In [ ]:
# 冒烟：学习层与 RL 层的快速测试（约 1 分钟）
!python -m pytest -q tests/test_learning.py tests/test_rl.py

## 阶段 B：GNN 动态模型

本地默认 400 个世界 × 24 步、20 轮；这里放大到 3000 个世界、40 轮、隐层 128。一半世界江湖化（`--jianghu 0.5`）。
报告里重点看：`changed_recall`（变化召回，基线 0）、`unchanged_kept`（未变保持，基线 1）、
`success_by_op`（动手的胜负带随机，看 `success_brier` 校准）、`attr_changed_by_attr`（伤 / 毒 / 制是否学会）。

In [ ]:
!python -m tianlong.learning.train --view env   --worlds 3000 --epochs 40 --hidden 128 --out artifacts/colab

In [ ]:
!python -m tianlong.learning.train --view agent --worlds 3000 --epochs 40 --hidden 128 --out artifacts/colab

## 阶段 C：RL 角色策略

模仿学习初始化 → PPO。Colab 免费版只有 2 个 CPU：采样进程开 1 个，学习器用 GPU。
`--predictor-path` 让策略以 GNN 的预测作为候选特征（不传则用启发式预测器），报告里的消融会回答“世界模型特征到底有没有被用上”。
评测指标：`goal_rate`、`false_accusations_per_ep`（冤枉人）、`attacks_per_ep`（奖励不禁止动手，策略学没学会“制住再搜”要如实看）。

In [ ]:
!python -m tianlong.learning.rl.train --demo-episodes 600 --bc-epochs 12 --bc-smoothing 0.1 \
    --ppo-iterations 120 --train-batch 6000 --entropy 0.02 --eval-episodes 200 \
    --env-runners 1 --gpus 1 --predictor-path artifacts/colab/dynamics_agent.pt --out artifacts/colab

In [ ]:
import json, pathlib
for f in sorted(pathlib.Path('artifacts/colab').glob('*.json')):
    print('=' * 20, f.name)
    print(json.dumps(json.loads(f.read_text()), indent=1, ensure_ascii=False)[:3000])

## 取回产物

下载 `tianlong_artifacts.zip`，在本地仓库解压到 `artifacts/`，然后：

```bash
python -m tianlong --predictor gnn --policy learned --artifacts artifacts/colab
```

In [ ]:
!cd artifacts && zip -qr ../tianlong_artifacts.zip colab && cd .. && ls -lh tianlong_artifacts.zip
try:
    from google.colab import files
    files.download('tianlong_artifacts.zip')
except Exception as e:
    print('非 Colab 环境，产物在', os.path.abspath('tianlong_artifacts.zip'))
# 也可以存到 Drive：
# from google.colab import drive; drive.mount('/content/drive')
# !cp tianlong_artifacts.zip /content/drive/MyDrive/